# Data quality and leakage
Check saved inputs before training. Held-out answer columns stay unread: only their file hashes, schemas and participant IDs are checked. No data or splits are changed.


In [1]:
import hashlib
import json
import re
import unicodedata
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())

def read_json(name):
    return json.loads((ROOT / name).read_text())

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

checks = []
def check(name, condition):
    passed = bool(condition)
    checks.append({"check": name, "passed": passed})
    assert passed, name

# Hash all data before checking it, including held-out files (no answer decoding).
before = {str(p.relative_to(ROOT)): sha256(p) for p in sorted((ROOT / "data").rglob("*")) if p.is_file()}
source = read_json("configs/data_source.json")
plan = read_json("configs/data_split_plan.json")
split = read_json("data/splits/question_split.json")
respondents = read_json("data/splits/respondent_split.json")
people = respondents["groups"]
manifest = read_json("data/splits/partition_manifest.json")
cleaning = read_json("outputs/response_cleaning_audit.json")
bank = read_json("data/processed/question_bank.json")
items = {i["item_id"]: i for i in bank["items"]}
train = set(split["training_columns"])
anchors = [q for group in split["anchor_groups"].values() for q in group]
validation = set(split["validation_policy"]["columns"] + split["validation_maximization"]["columns"])
test = set(split["final_test"]["columns"])
forbidden = set(split["forbidden_input_columns_all_respondents_all_waves"])


## 1. Provenance and fixed splits


In [2]:
check("source manifests agree", source == read_json("data/raw/twin_2k_500/source_manifest.json"))
for f in source["files"]:
    check("source checksum: " + f["repo_path"], sha256(ROOT / f["local_path"]) == f["sha256"])
for p in manifest["partitions"]:
    check("partition checksum: " + p["partition"], sha256(ROOT / p["path"]) == p["sha256"])
for p in cleaning["partitions"]:
    check("cleaned checksum: " + p["partition"], sha256(ROOT / p["output_path"]) == p["output_sha256"])
check("question bank checksum", sha256(ROOT / "data/processed/question_bank.json") == cleaning["question_bank_sha256"])
check("question split checksum", sha256(ROOT / "data/splits/question_split.json") == bank["question_split_sha256"])
check("split matches plan", split == plan["question_split"])
check("source revision matches", all(d["source_revision"] == source["revision"]
                                    for d in [bank, split, respondents, manifest, plan, cleaning]))
check("participant counts", {g: len(ids) for g, ids in people.items()} == plan["respondent_counts"])
check("recorded seeds agree", respondents["seed"] == manifest["seed"] == plan["seed"])
check("unique participants within each split", all(len(ids) == len(set(ids)) for ids in people.values()))
check("disjoint participant groups", all(not set(people[a]) & set(people[b]) for a, b in combinations(people, 2)))
all_ids = sorted(pid for ids in people.values() for pid in ids)
expected = np.random.default_rng(plan["seed"]).permutation(all_ids).tolist()
check("seeded participant split reproduced", expected == sum(people.values(), []))
assignments = pd.read_csv(ROOT / "data/splits/respondent_assignments.csv", dtype={"pid": "string"})
expected_assignments = pd.DataFrame([
    {"pid": pid, "split": group, "position_in_split": position}
    for group, ids in people.items() for position, pid in enumerate(ids)
]).astype({"pid": "string"})
pd.testing.assert_frame_equal(assignments, expected_assignments)
check("CSV participant assignments agree", True)
question_assignments = pd.read_csv(ROOT / "data/splits/question_assignments.csv")
check("unique CSV question assignments", question_assignments["answer_column"].is_unique)
for role, ids in [("shared_training", train), ("validation_policy_only", set(split["validation_policy"]["columns"])),
                  ("validation_maximization_only", set(split["validation_maximization"]["columns"])), ("final_test_only", test)]:
    check("CSV question assignments: " + role,
          set(question_assignments.loc[question_assignments["role"].eq(role), "answer_column"]) == ids)
check("CSV anchor assignments", set(question_assignments.loc[question_assignments["persona_evidence"], "answer_column"]) == set(anchors))
check("question sets disjoint", not (train & validation or train & test or validation & test))
check("92 unique permitted anchor items", len(anchors) == len(set(anchors)) == 92 and set(anchors) <= train)
check("forbidden questions absent from inputs", not train & forbidden and validation | test <= forbidden)
check("perceived policy support excluded", not any(q.startswith("QID290") for q in train))
check("supplied persona summaries excluded", split["exclude_supplied_persona_summaries"] and
      not any("persona" in p.lower() for p in before if p.startswith("data/raw/")))
check("bank contains precisely selected questions", set(items) == train | validation | test and len(items) == len(bank["items"]))
for role, ids in [("training", train), ("validation", validation), ("test", test)]:
    check("bank role: " + role, {q for q, item in items.items() if item["role"] == role} == ids)
check("bank anchor flags", {q for q, item in items.items() if item["is_anchor"]} == set(anchors))
print("Source hashes, split membership and question exclusions passed.")


Source hashes, split membership and question exclusions passed.


## 2. Question wording and answer mappings


In [3]:
catalogue_file = next(f for f in source["files"] if f["repo_path"].endswith("/question_catalog.json"))
catalogue = json.loads((ROOT / catalogue_file["local_path"]).read_text())
def source_key(record):
    return "::".join([record["BlockName"].strip(), record["QuestionType"], record["QuestionID"]])
originals = {source_key(s): s for s in catalogue}
check("original catalogue records preserved", all(originals[k] == v for k, v in bank["sources"].items()))
for item in items.values():
    parent = originals[item["parent_key"]]
    assert item["stem"] == parent["QuestionText"]
    if item["row_position"] is not None:
        row = item["row_position"] - 1
        assert parent["Rows"][row] == item["row_text"]
        assert parent["csv_columns"][row] == item["item_id"]
    for instruction in item["instructions"]:
        assert instruction["text"] == originals[instruction["source_key"]]["QuestionText"][instruction["start"]:instruction["end"]]
    original_options = parent["Columns"] if parent["QuestionType"] == "Matrix" else parent["Options"]
    resolved = item["row_text"].split(":", 1) if item["response_format"] == "bipolar_choice" else original_options
    assert [o["source_label"] for o in item["options"]] == original_options
    assert [o["label"] for o in item["options"]] == resolved
    assert [o["code"] for o in item["options"]] == list(range(1, len(resolved) + 1))
    assert [o["index"] for o in item["options"]] == list(range(len(resolved)))
    assert all(label.strip() for label in resolved) and len(set(resolved)) == len(resolved)
    parts = []
    if item["instructions"]:
        parts.append("Instructions:\n" + "\n\n".join(c["text"] for c in item["instructions"]))
    parts.append("Question:\n" + item["stem"])
    if item["matrix_context_rows"]:
        assert item["matrix_context_rows"] == parent["Rows"]
        parts.append("Full list (answer only the indicated row):\n" + "\n".join(item["matrix_context_rows"]))
    if item["row_text"] is not None:
        parts.append("Indicated row:\n" + item["row_text"])
    parts.append("Answer options:\n" + "\n".join(f"{o['code']}. {o['label']}" for o in item["options"]))
    assert item["question_text"] == "\n\n".join(parts)
check("all 441 question texts and option mappings verified", True)


## 3. Clean response tables
Check every permitted row. Missing answers stay missing; identical complete evidence profiles are flagged, not automatically removed.


In [4]:
summaries, evidence = [], []
partition_by_name = {p["partition"]: p for p in manifest["partitions"]}
groups = {"shared_training": "shared_training", "development_anchors": "development", "final_anchors": "final_personas"}
for p in cleaning["partitions"]:
    name = p["partition"]
    group = groups[name]
    columns = split["training_columns"] if group == "shared_training" else anchors
    wide = pd.read_parquet(ROOT / p["input_path"])
    long = pd.read_parquet(ROOT / p["output_path"])
    check(name + ": exact input columns and people", wide.columns.tolist() == ["pid"] + columns and wide["pid"].tolist() == people[group])
    check(name + ": unique complete person/item grid", not long.duplicated(["pid", "item_id"]).any() and
          len(long) == len(people[group]) * len(columns) and set(long["pid"]) == set(people[group]) and set(long["item_id"]) == set(columns))
    check(name + ": permitted wave and role", long["source_wave"].eq("wave1_3").all() and
          long["question_role"].eq("training").all() and long["respondent_group"].eq(group).all() and not set(long["item_id"]) & forbidden)
    for field in ["response_code", "answer_index", "answer_text", "source_option_label"]:
        check(name + ": missing " + field, long[field].isna().equals(long["is_missing"]))
    observed = long.loc[~long["is_missing"]]
    check(name + ": valid codes and indices", observed["response_code"].between(1, observed["n_options"]).all() and
          (observed["answer_index"] == observed["response_code"] - 1).all())
    for q, rows in long.groupby("item_id", observed=True):
        item = items[str(q)]
        assert rows["question_text"].eq(item["question_text"]).all()
        assert rows["n_options"].eq(len(item["options"])).all()
        assert rows["is_anchor"].eq(item["is_anchor"]).all()
        assert rows["parent_key"].eq(item["parent_key"]).all()
        answers = rows.loc[~rows["is_missing"]]
        for field, key in [("answer_text", "label"), ("source_option_label", "source_label")]:
            lookup = {o["code"]: o[key] for o in item["options"]}
            assert answers[field].astype("string").equals(answers["response_code"].map(lookup).astype("string"))
    check(name + ": question and selected option join", True)
    restored = long.pivot(index="pid", columns="item_id", values="response_code").reindex(index=people[group], columns=columns)
    pd.testing.assert_frame_equal(restored.rename_axis(columns=None), wide.set_index("pid"), check_dtype=False, check_index_type=False, check_column_type=False)
    check(name + ": lossless wide-to-long conversion", True)
    observed_per_person = wide[columns].notna().sum(axis=1)
    check(name + ": each person and item has evidence", observed_per_person.gt(0).all() and wide[columns].notna().any().all())
    missing = long.loc[long["is_missing"]].groupby("item_id", observed=True).size()
    summaries.append({"partition": name, "people": len(wide), "items": len(columns), "rows": len(long),
                      "missing_answers": int(long["is_missing"].sum()),
                      "missing_percent": round(float(long["is_missing"].mean() * 100), 4),
                      "minimum_answers_per_person": int(observed_per_person.min()),
                      "missing_by_item": {str(q): int(n) for q, n in missing.items()},
                      "constant_items": [q for q in columns if wide[q].nunique() <= 1]})
    evidence.append(wide.set_index("pid")[anchors].assign(respondent_group=group))
    del long, observed

profiles = pd.concat(evidence)
duplicates = profiles.duplicated(subset=anchors, keep=False)
duplicate_profiles = {"people_with_identical_92_item_profiles": int(duplicates.sum()), "cross_split_groups": 0}
if duplicates.any():
    grouped = profiles.loc[duplicates].groupby(anchors, dropna=False)["respondent_group"].nunique()
    duplicate_profiles["cross_split_groups"] = int(grouped.gt(1).sum())
display(pd.DataFrame(summaries)[["partition", "people", "items", "rows", "missing_answers", "minimum_answers_per_person"]])
print(duplicate_profiles)


,partition,people,items,rows,missing_answers,minimum_answers_per_person
0,shared_training,1658,430,712940,113,418
1,development_anchors,300,92,27600,24,85
2,final_anchors,100,92,9200,2,91


{'people_with_identical_92_item_profiles': 0, 'cross_split_groups': 0}


## 4. Held-out boundaries
Read only IDs and Parquet metadata. Earlier-wave policy answers remain excluded for every person. Answer validity was checked during partitioning; matching hashes preserve that check without reopening labels.


In [5]:
heldout_specs = {
    "validation_policy": ("development", split["validation_policy"]["columns"]),
    "validation_maximization": ("development", split["validation_maximization"]["columns"]),
    "final_test": ("final_personas", split["final_test"]["columns"]),
    "final_retest_reference": ("final_personas", split["final_test"]["columns"]),
}
for name, (group, columns) in heldout_specs.items():
    path = ROOT / partition_by_name[name]["path"]
    parquet = pq.ParquetFile(path)
    check(name + ": schema and row count", parquet.schema_arrow.names == ["pid"] + columns and parquet.metadata.num_rows == len(people[group]))
    ids = parquet.read(columns=["pid"]).column("pid").to_pylist()
    check(name + ": expected participant IDs", ids == people[group])
print("Four held-out files verified; no held-out answer columns decoded.")


Four held-out files verified; no held-out answer columns decoded.


## 5. Question overlap review
Exact matching checks full training context. Word/character similarity suggests review candidates; it cannot establish semantic equivalence. The recorded review also considers related policy, environmental-value and personality items. No answers are used.


In [6]:
def normalize(text):
    return " ".join(re.findall(r"\w+", unicodedata.normalize("NFKC", text).casefold()))

training_items = [items[q] for q in sorted(train)]
heldout_items = [items[q] for q in sorted(validation | test)]
exact_matches = []
for held in heldout_items:
    target = normalize(held["row_text"] or held["stem"])
    for known in training_items:
        if target in normalize(known["question_text"]):
            exact_matches.append([held["item_id"], known["item_id"]])
check("no held-out statement in rendered training context", not exact_matches)
check("no test statement in validation context", all(
    normalize(items[t]["row_text"] or items[t]["stem"]) not in normalize(items[v]["question_text"])
    for t in test for v in validation))

# Use statement text to avoid generic survey instructions dominating the ranking.
# For bipolar choices retain the stem: it defines what 'Lottery' means.
def comparison_text(item):
    if item["response_format"] == "bipolar_choice":
        return item["stem"] + " " + item["row_text"]
    return item["row_text"] or item["stem"]
texts = [comparison_text(i) for i in training_items + heldout_items]
candidates = []
for method, settings in [("word", {"stop_words": "english", "ngram_range": (1, 2)}),
                         ("character", {"analyzer": "char_wb", "ngram_range": (3, 5)})]:
    vectors = TfidfVectorizer(**settings).fit_transform(texts)
    scores = (vectors[len(training_items):] @ vectors[:len(training_items)].T).toarray()
    for row, held in enumerate(heldout_items):
        for rank, j in enumerate(np.argsort(-scores[row], kind="stable")[:3], start=1):
            known = training_items[j]
            candidates.append({"heldout_item": held["item_id"], "training_item": known["item_id"],
                               "method": method, "rank": rank, "similarity": round(float(scores[row, j]), 4),
                               "heldout_statement": comparison_text(held), "training_statement": comparison_text(known)})

# Assistant text review, tied to this exact bank. Re-review if wording changes.
check("review applies to this bank", sha256(ROOT / "data/processed/question_bank.json") ==
      "a1a83bd0ec96873a7c5f5d793b35cd54e7d23f81e1cf52eca8dd3ff01eb7b941")
review_notes = {
    "QID239_1": "Radio-station maximization; competitiveness and financial vignettes do not ask this behaviour.",
    "QID239_2": "Seeking better jobs despite satisfaction; job diligence and ambition are related but distinct.",
    "QID239_3": "Difficulty choosing a gift; empathy for friends is not the same preference.",
    "QID239_4": "Difficulty finding ideal clothes; spending restraint and general indecision are distinct.",
    "QID239_5": "Highest personal standards; achievement, diligence and accuracy are related constructs, not equivalent statements.",
    "QID239_6": "Never settling for second best; competitiveness and superiority are related, not synonymous across contexts.",
    "QID287_1": "Carbon-tax support; environmental values are permitted predictors. The IRS vignette asks emotional framing, not policy support.",
    "QID287_2": "40% clean-energy allocation to low-income communities; income, equality and environmental values do not repeat this policy.",
    "QID287_3": "Federal carbon-free-electricity investment by 2035; general environmental values do not repeat this policy.",
    "QID287_7": "Mandatory paid parental leave; family obligations and other policy preferences are distinct.",
    "QID287_10": "2% wealth tax above $50m; personal income, wealth values and tax-loss vignettes are distinct.",
}
check("all 11 held-out items reviewed", set(review_notes) == validation | test)
shared_parents = sorted({i["parent_key"] for i in training_items} & {i["parent_key"] for i in heldout_items})
check("only planned policy parent crosses question splits", shared_parents == ["False consensus::Matrix::QID287"])
display(pd.DataFrame(candidates).query("method == 'word' and rank == 1")[["heldout_item", "training_item", "similarity"]])
print("No equivalent held-out question identified in the text review. Related constructs are retained.")


,heldout_item,training_item,similarity
0,QID239_1,QID152,0.0647
3,QID239_2,QID233_5,0.1397
6,QID239_3,QID232_3,0.1037
9,QID239_4,QID26_7,0.1258
12,QID239_5,QID27_5,0.1379
15,QID239_6,QID17,0.2304
18,QID287_1,QID150,0.0764
21,QID287_10,QID150,0.0604
24,QID287_2,QID25_11,0.2041
27,QID287_3,QID29_20,0.0766


No equivalent held-out question identified in the text review. Related constructs are retained.


## 6. Save the audit
This audit permits feature generation, not a claim of predictive accuracy. Training must load explicit permitted tables, fit learned preprocessing on training questions only, and keep final labels sealed until the model and evaluation rules are frozen.


In [7]:
check("all input data unchanged", before == {str(p.relative_to(ROOT)): sha256(p)
      for p in sorted((ROOT / "data").rglob("*")) if p.is_file()})
limitations = [
    "The policy matrix intentionally spans roles; different policy rows are held out, not the whole survey instrument.",
    "Semantic review is assistant judgement supported by lexical screening, not proof that no subtle proxy exists.",
    "Raw data and provenance source records include held-out content. Model inputs must use permitted long tables and per-item question_text, never entire raw files or bank.sources.",
    "Public-dataset contamination of a future pretrained encoder is unknown and cannot be ruled out by this audit.",
    "The random panel is a historical Prolific cohort; it is not a weighted sample of current US opinion.",
    "Repeated matrix rows are correlated; rows are not independent observations. Use person/item-aware evaluation.",
    "Missing answers remain null and must be masked during fitting. Constant items in a small panel are reported, not automatically dropped.",
]
if duplicate_profiles["people_with_identical_92_item_profiles"]:
    limitations.append("Identical evidence profiles were detected; inspect before treating these as distinct independent people.")
report = {
    "status": "passed_with_caveats", "automated_checks_passed": len(checks),
    "source_revision": source["revision"], "checks": checks,
    "response_quality": summaries, "duplicate_evidence_profiles": duplicate_profiles,
    "semantic_review": {"reviewer": "Codex assistant", "bank_sha256": sha256(ROOT / "data/processed/question_bank.json"),
                        "finding": "No equivalent held-out question identified; related predictors retained.",
                        "validation_test_review": "Paid leave and wealth-tax validation policies are distinct from the three climate test policies; shared tax/redistribution themes are intentional transfer, not equivalent outcomes.",
                        "exact_matches": exact_matches, "shared_parent_questions": shared_parents,
                        "item_reviews": review_notes, "lexical_candidates": candidates},
    "heldout_access": "SHA-256, Parquet schema/row count, and pid column only; no answer columns decoded.",
    "heldout_answer_columns_loaded": False, "data_modified": False, "training_started": False,
    "input_sha256": before, "limitations": limitations,
    "next_step": "Generate frozen language-model features; fit PCA/scalers on training questions only."
}
destination = ROOT / "outputs/data_quality_leakage_audit.json"
destination.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(f"Passed {len(checks)} automated checks. Saved {destination.relative_to(ROOT)}")


Passed 93 automated checks. Saved outputs/data_quality_leakage_audit.json
